# 3. Pós-Processamento e Validação de Negócio

## Introdução e Contextualização

O objetivo desta etapa final de pós-processamento é traduzir os resultados matemáticos da nossa tarefa de Detecção de Outlier (obtidos via K-Means e HDBSCAN) em inteligência acionável. Em projetos de ciência de dados aplicados a auditoria e controle social, não basta apenas identificar instâncias atípicas no espaço vetorial; é imprescindível cruzar essas instâncias anômalas (neste caso, os CPFs ou Unidades Gestoras) com a base original para investigar possíveis fraudes ou abusos no uso do Cartão de Pagamento do Governo Federal (CPGF).

Ao integrar as pontuações de anomalia geradas pelos modelos não supervisionados aos dados categóricos vitais, criamos um mecanismo de ranqueamento que permite direcionar o esforço de auditoria manual para os casos com maior evidência de comportamento divergente do padrão global.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configurações de exibição e supressão de avisos para um notebook limpo
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# 2. Carga de Dados e Preparação do Merge
print("Carregando a base original e a base minerada...")

# Carregamento da base original com os metadados das transações
# Tratamos eventuais problemas de separador comumente encontrados nos CSVs
cpgf_limpo = pd.read_csv('../data/cpgf_limpo.csv', sep=';', encoding='utf-8')

# Carregamento da base com os resultados da mineração (scores de anomalia)
# Garantindo que o índice seja o 'CPF PORTADOR'
cpgf_minerado = pd.read_csv('../data/cpgf_minerado.csv', sep=';', encoding='utf-8', index_col=0)

print(f"Linhas em cpgf_limpo: {cpgf_limpo.shape[0]}")
print(f"CPFs processados em cpgf_minerado: {cpgf_minerado.shape[0]}")

## Definição do Threshold e Filtro de Anomalias

Para selecionar de forma rigorosa as instâncias que devem compor nosso dossiê de auditoria, estabelecemos um limite empírico de severidade (*threshold*). Com base na análise de densidade da cauda longa da distribuição dos scores (etapa visualizada na modelagem HDBSCAN), observou-se que as anomalias de altíssima significância estatística se isolam na porção final da curva.

Por essa razão, o *threshold* adotado para o `HDBSCAN_Outlier_Score` será de **0.80**. Isso significa que filtraremos apenas as entidades geradoras de despesas cujo perfil de gastos diverge substancialmente (> 80% de score de anomalia) do macrogrupo.

In [ ]:
# 3. Definição do Threshold e Filtro de Anomalias

# Limite empírico de severidade definido a partir da distribuição dos scores
THRESHOLD = 0.80

# Isolando os CPFs (índices) que ultrapassam o threshold
alvos_auditoria = cpgf_minerado[cpgf_minerado['HDBSCAN_Outlier_Score'] >= THRESHOLD].copy()

print(f"Total de CPFs acima do *threshold* ({THRESHOLD}): {alvos_auditoria.shape[0]}")

# O campo de cruzamento na base original é 'CPF PORTADOR'
# Iremos trazer as colunas de Score e Distância para as transações da base limpa
df_investigacao = pd.merge(
    alvos_auditoria[['HDBSCAN_Outlier_Score', 'Distancia_Centroide_KMeans']],
    cpgf_limpo,
    left_index=True,
    right_on='CPF PORTADOR',
    how='left'
)

print(f"Transações associadas aos CPFs anômalos: {df_investigacao.shape[0]}")

## Análise Qualitativa: Dossiê de Auditoria

Para demonstrar o viés prático e o retorno de valor (ROI) do projeto, extrairemos o **Top 10 CPFs mais críticos** (baseado no maior `HDBSCAN_Outlier_Score`) para uma análise manual detalhada.

Essa listagem funciona como um Dossiê de Auditoria: o algoritmo aponta as maiores suspeitas com base em critérios multicritério (valores, frequência e perfil de estabelecimentos) e consolida um resumo financeiro e de faturamento (as 5 maiores compras). Esta é uma ferramenta gerencial direta para as equipes de controle (CGU, TCU, etc.).

In [ ]:
from IPython.display import display

# 4. Análise Qualitativa: Dossiê de Auditoria

# Identificar os 10 CPFs com maior Score de Anomalia via HDBSCAN
top_10_cpfs = alvos_auditoria.sort_values(by='HDBSCAN_Outlier_Score', ascending=False).head(10)

print("=" * 80)
print(" " * 20 + "DOSSIÊ DE AUDITORIA: TOP 10 CPFS SUSPEITOS")
print("=" * 80 + "\n")

for cpf_portador, row in top_10_cpfs.iterrows():
    score = row['HDBSCAN_Outlier_Score']
    
    # Filtrar todas as transações relativas a este CPF
    transacoes_cpf = df_investigacao[df_investigacao['CPF PORTADOR'] == cpf_portador]
    
    # Alguns gastos são sigilosos, e o CPF pode estar mascarado ou agrupado apenas pela Unidade Gestora
    # Pegaremos o nome da Unidade Gestora como referência adicional caso o CPF seja genérico (ex: Sigiloso)
    unidade_gestora = transacoes_cpf['NOME UNIDADE GESTORA'].iloc[0] if not transacoes_cpf.empty else 'DESCONHECIDA'
    
    # Calcular o valor total gasto pelo CPF entre as transações anômalas
    # Como a base pode ter valores em string ou float dependendo do parser, garantimos a conversão
    if 'VALOR TRANSAÇÃO' in transacoes_cpf.columns:
        # Se for string com vírgula, tratar, senão considerar float
        if transacoes_cpf['VALOR TRANSAÇÃO'].dtype == 'O':
            transacoes_cpf['VALOR TRANSAÇÃO'] = transacoes_cpf['VALOR TRANSAÇÃO'].astype(str).str.replace(',', '.').astype(float)
        valor_total = transacoes_cpf['VALOR TRANSAÇÃO'].sum()
    else:
        valor_total = 0.0
        
    print(f"➔ ALVO: {cpf_portador} | UNIDADE GESTORA: {unidade_gestora}")
    print(f"   Score de Anomalia (HDBSCAN): {score:.4f}")
    print(f"   Volume Total Envolvido nas Transações: R$ {valor_total:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
    print(f"   Top 5 Maiores Gastos Individuais:")
    
    # Extrair as 5 maiores compras para compor as evidências primárias
    colunas_evidencia = ['DATA TRANSAÇÃO', 'NOME FAVORECIDO', 'VALOR TRANSAÇÃO']
    # Verificando se todas as colunas necessárias existem no DF
    colunas_presentes = [col for col in colunas_evidencia if col in transacoes_cpf.columns]
    
    if not transacoes_cpf.empty and len(colunas_presentes) == len(colunas_evidencia):
        maiores_compras = transacoes_cpf.nlargest(5, 'VALOR TRANSAÇÃO')[colunas_evidencia]
        
        # Formatando o DataFrame para exibição otimizada
        maiores_compras['VALOR TRANSAÇÃO'] = maiores_compras['VALOR TRANSAÇÃO'].apply(lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
        maiores_compras.reset_index(drop=True, inplace=True)
        maiores_compras.index += 1  # Para começar de 1 o log
        
        display(maiores_compras)
    else:
        print("   [!] Dados insuficientes para detalhar as maiores compras.")
        
    print("-" * 80)

## Geração de Insights Gráficos para a Apresentação

A visualização a seguir tem como objetivo sintetizar os resultados no nível da **Unidade Gestora**. Isso permite identificar sistemicamente quais órgãos ou entidades centralizam o maior volume financeiro de transações sinalizadas como atípicas ou anômalas.

Este tipo de gráfico é uma ferramenta essencial de alto impacto para ser incorporada aos slides de defesa do projeto.

In [ ]:
# 5. Geração de Insights Gráficos para a Apresentação

if not df_investigacao.empty and 'VALOR TRANSAÇÃO' in df_investigacao.columns:
    # Agrupar as transações anômalas por Unidade Gestora, somando o valor
    agrupamento_ug = df_investigacao.groupby('NOME UNIDADE GESTORA')['VALOR TRANSAÇÃO'].sum().reset_index()
    
    # Selecionar o Top 10 com maior volume financeiro anômalo
    top_10_ug = agrupamento_ug.sort_values(by='VALOR TRANSAÇÃO', ascending=False).head(10)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=top_10_ug,
        x='VALOR TRANSAÇÃO',
        y='NOME UNIDADE GESTORA',
        palette='Reds_r' # Paleta de cores sugerindo calor/atenção/risco
    )
    
    plt.title('Top 10 Unidades Gestoras por Maior Volume Financeiro em Transações Anômalas', fontsize=14, fontweight='bold')
    plt.xlabel('Volume Financeiro Total (R$)', fontsize=12)
    plt.ylabel('Nome da Unidade Gestora', fontsize=12)
    
    # Formatação do eixo X para representar milhares/milhões conforme necessidade
    ax = plt.gca()
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"R$ {x:,.0f}".replace(',', '.')))
    
    plt.tight_layout()
    plt.show()
else:
    print("Não há dados suficientes ou a coluna 'VALOR TRANSAÇÃO' não existe para gerar o gráfico.")

## Exportação do Relatório Final

Consolidadas todas as evidências, exportaremos os dados na granularidade de transações. O arquivo será ordenado pelo nível de alerta estrutural (`HDBSCAN_Outlier_Score`), criando um entregável prático e tabulado `relatorio_auditoria_final.csv`, pronto para ser auditado, sem requerer que o analista de negócios possua conhecimentos em programação Python.

In [ ]:
# 6. Exportação do Relatório Final

# Ordenar o dataframe focado na investigação garantindo que as piores anomalias fiquem no topo
df_exportacao = df_investigacao.sort_values(by='HDBSCAN_Outlier_Score', ascending=False)

# Exportação do entregável prático
caminho_exportacao = '../data/relatorio_auditoria_final.csv'
df_exportacao.to_csv(caminho_exportacao, index=False, sep=';', encoding='utf-8')

print(f"Relatório de auditoria gerado com sucesso!")
print(f"Caminho do arquivo: {caminho_exportacao}")
print(f"Total de transações para auditoria processadas: {df_exportacao.shape[0]}")